## A short notebook tutorial on Spherical Harmonic (Ridge) Regression

Last Modified: July 23rd, 2024

Author: [Opal Issan](https://opaliss.github.io/opalissan/) (PhD student @UCSD). contact: oissan@ucsd.edu

In [ ]:
import sys, os

sys.path.append(os.path.abspath(os.path.join("..")))

In [ ]:
import numpy as np
import scipy
import cartopy.crs as ccrs
from supermag_api import *
from spherical_harmonics import (
    get_spherical_harmonic_basis_matrix,
    ridge_regression,
    construct_global_view,
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from mycolorpy import colorlist as mcp

font = {"family": "serif", "size": 14}

matplotlib.rc("font", **font)
matplotlib.rc("xtick", labelsize=14)
matplotlib.rc("ytick", labelsize=14)

# SuperMAG data

In [ ]:
# start date year-month-day-hour-min-sec
# St. Patricks Day Storm
start = [2015, 3, 17, 18, 40, 10]

In [ ]:
# read in data
status, stations = SuperMAGGetInventory("opaliss", start, 3600)
# number of stations
N = len(stations)
print("number of stations = ", N)

In [ ]:
# intialize data
data_Bn = np.zeros(N)
geo_lat = np.zeros(N)
geo_lon = np.zeros(N)

In [ ]:
# read in data for 1hr in advance from start date at station "res"
# note: this is very slow.. not sure if there are faster ways to go about this
kk = 0
for ii in range(0, N):
    status, sm_data = SuperMAGGetData("opaliss", start, 3600, "geo", stations[ii])
    try:
        data_Bn[kk] = sm_data.N[0]["geo"]
        geo_lat[kk] = sm_data.glat[0] + 90
        geo_lon[kk] = sm_data.glon[0]
        kk += 1
    except:
        print("An exception occurred at " + str(stations[ii]))
data_Bn = data_Bn[:kk]
geo_lat = geo_lat[:kk]
geo_lon = geo_lon[:kk]

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mollweide())

ax.stock_img()
plt.scatter(geo_lon, geo_lat - 90, color="red", s=10, transform=ccrs.PlateCarree())

_ = plt.title("SuperMAG stations")
ax.scatter(0, 0, s=2)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()

# plt.savefig("figures/supermag_stations_locations.png", bbox_inches='tight', dpi=600)

## Spherical Harmonics Regression 

In [ ]:
# order of spherical harmonic interpolator
ell = 12
# ridge regression regularization
lambda_ = 0.1

In [ ]:
Y = get_spherical_harmonic_basis_matrix(
    latitude=geo_lat * np.pi / 180, longitude=geo_lon * np.pi / 180, ell=ell
)

In [ ]:
pred = construct_global_view(
    coeff=ridge_regression(basis_matrix=Y, data=data_Bn, lambda_=lambda_),
    longitude=geo_lon * np.pi / 180,
    latitude=geo_lat * np.pi / 180,
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pred, label="Spherical Harmonic Interpolation", linewidth=2, color="blue")
ax.plot(data_Bn, ls=":", label="SuperMAG", linewidth=2, color="red")
ax.set_xlabel("observation index")
ax.set_ylabel("Bn [nT]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.set_title("$\ell = $" + str(ell))
_ = plt.tight_layout()
_ = plt.legend()

In [ ]:
N_interp = 100
THETA, PHI = np.meshgrid(
    np.linspace(0, 180, N_interp, endpoint=False),
    np.linspace(0, 360, N_interp, endpoint=False),
)

In [ ]:
pred = construct_global_view(
    coeff=ridge_regression(basis_matrix=Y, data=data_Bn, lambda_=lambda_),
    longitude=np.ndarray.flatten(PHI) * np.pi / 180,
    latitude=np.ndarray.flatten(THETA) * np.pi / 180,
)

In [ ]:
fig = plt.figure(figsize=(9, 4))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.coastlines()
pos = ax.pcolormesh(
    PHI,
    THETA - 90,
    np.reshape(pred, (N_interp, N_interp), order="C"),
    vmin=-350,
    vmax=350,
    alpha=0.5,
    transform=ccrs.PlateCarree(),
)

plt.scatter(
    geo_lon,
    geo_lat - 90,
    c=data_Bn,
    s=10,
    cmap="viridis",
    vmin=-350,
    vmax=350,
    transform=ccrs.PlateCarree(),
)

cbar = fig.colorbar(pos)
cbar.ax.set_ylabel("$B_{n}$ [nT]", rotation=90)

ax.set_title("Spherical Harmonic Interpolation")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()

# plt.savefig("figures/supermag_stations_interpolation.png", bbox_inches='tight', dpi=600)

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mollweide())

ax.coastlines()
pos = ax.pcolormesh(
    PHI,
    THETA - 90,
    np.reshape(pred, (N_interp, N_interp), order="C"),
    vmin=-350,
    vmax=350,
    alpha=0.4,
    transform=ccrs.PlateCarree(),
)

plt.scatter(
    geo_lon,
    geo_lat - 90,
    c=data_Bn,
    s=10,
    cmap="viridis",
    vmin=-350,
    vmax=350,
    transform=ccrs.PlateCarree(),
)

cbar = fig.colorbar(pos, orientation="horizontal")
cbar.ax.set_ylabel("$B_{n}$ [nT]", rotation=90)


_ = plt.title("SuperMAG Spherical Harmonic Interpolation")
ax.scatter(0, 0, s=2)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()

# plt.savefig("figures/supermag_stations_locations.png", bbox_inches='tight', dpi=600)